In [ ]:
# 03. 파생변수 → 튜닝 → 앙상블 (Train 5-fold CV, 반복 CV 5×3)
import sys, warnings
sys.path.insert(0, "../src"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from steel import *
pd.set_option("display.width", 200); pd.set_option("display.precision", 4)
import json
from itertools import combinations
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import RepeatedStratifiedKFold

X_train, X_test, y_train, y_test = load_data()
XF = add_features(X_train)   # 행 단위 파생변수 8개 → 34열, 강판 맥락 2개는 Pipeline 안에서 추가 → 모델 입력 36개
print(X_train.shape[1], "→", len(fe_features(XF.columns)), "features")

In [ ]:
# 파생변수 효과: 같은 GBM 탐색 공간으로 원본 26개 vs 36개 각각 튜닝
from sklearn.ensemble import GradientBoostingClassifier
def gbm26():
    pipe, n, space = search_spaces(XF.columns, models=("GBM",))["GBM"]
    pipe26 = make_pipeline(BalancedWeightClassifier(GradientBoostingClassifier(random_state=RS), alpha=1), X_train.columns)
    return tune("GBM", (pipe26, n, space), X_train, y_train)
r26 = cached("tune_gbm_26", gbm26)
r36 = cached("tune_gbm_36", lambda: tune("GBM", search_spaces(XF.columns, models=("GBM",))["GBM"], XF, y_train))
pd.DataFrame({"원본 26개": [r26["score"], r26["std"]], "+파생변수 36개": [r36["score"], r36["std"]]}, index=["macro_f1", "std"])
# → 파생변수 추가 + 재튜닝으로 개선 (파라미터를 고정한 채 변수만 추가하면 효과가 거의 없었음)

In [ ]:
# 5개 모델을 36개 피처로 튜닝 (샘플러 SMOTENC/없음, 클래스 가중치도 탐색 대상)
SPACES = search_spaces(XF.columns)
TUNED = {n: cached(f"tune_{n}", lambda n=n: tune(n, SPACES[n], XF, y_train)) for n in SPACES}
pd.DataFrame({n: {"macro_f1": r["score"], "std": r["std"]} for n, r in TUNED.items()}).T.sort_values("macro_f1", ascending=False)

In [ ]:
# 튜닝 모델별 OOF 확률 → Soft Voting 조합 비교 (확률 평균)
CLS = np.array(CLASS_ORDER)
model = lambda n: clone(SPACES[n][0]).set_params(**TUNED[n]["params"])
def oof_proba(splits):
    def one(n, i, tr, va):
        m = model(n).fit(XF.iloc[tr], y_train.iloc[tr])
        return n, i, pd.DataFrame(m.predict_proba(XF.iloc[va]), columns=m.classes_)[CLASS_ORDER].values
    return {(n, i): p for n, i, p in Parallel(n_jobs=-1)(delayed(one)(n, i, tr, va) for n in SPACES for i, (tr, va) in enumerate(splits))}
def scores(members, P, splits):   # fold별 Macro F1
    return np.array([f1_score(y_train.iloc[va], CLS[np.mean([P[(n, i)] for n in members], axis=0).argmax(1)], average="macro")
                     for i, (_, va) in enumerate(splits)])

S5 = list(CV.split(XF, y_train))
P5 = cached("oof_5fold", lambda: oof_proba(S5))
combos = [c for r in range(1, 6) for c in combinations(SPACES, r)]
vote = pd.Series({" + ".join(c): scores(c, P5, S5).mean() for c in combos}).sort_values(ascending=False)
vote.head(8).to_frame("5-fold macro_f1")
# → GBM이 포함된 배깅+부스팅 조합이 상위. 부스팅끼리 묶은 조합은 GBM 단독보다 낮음

In [ ]:
# 상위 후보를 반복 CV(5×3, 15 fold)로 재확인 → 1번 분할의 운을 배제
SR = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RS).split(XF, y_train))
PR = cached("oof_repeated", lambda: oof_proba(SR))
CANDS = {"GBM": ["GBM"], "ExtraTrees + GBM": ["ExtraTrees", "GBM"], "ExtraTrees + RandomForest + GBM": ["ExtraTrees", "RandomForest", "GBM"]}
rep = {k: scores(v, PR, SR) for k, v in CANDS.items()}
pd.DataFrame({k: {"5-fold": scores(v, P5, S5).mean(), "repeated_mean": rep[k].mean(), "repeated_std": rep[k].std(),
                  "GBM보다 좋은 fold": f"{(rep[k] > rep['GBM']).sum()}/15"} for k, v in CANDS.items()}).T
# → ET+GBM과 ET+RF+GBM은 사실상 동률 (차이 < 0.001) → 다양성이 큰 3개 조합을 최종 모델로 선택

In [ ]:
# 최종 구성 모델 파라미터 저장 (src/steel.py의 FINAL_PARAMS와 동일)
final = {n: {k: v for k, v in TUNED[n]["params"].items() if k != "sampler"} for n in ["ExtraTrees", "RandomForest", "GBM"]}
json.dump(final, open("../results/final_params.json", "w"), indent=2, default=float)
final